# Import Tables from Parquet

This notebook imports .parquet files from the 'import' volume and creates Delta tables identical to the original setup.

Use this as an alternative to running the individual download notebooks when you already have the parquet exports.

In [ ]:
# Configuration
CATALOG = "main_catalog"
SCHEMA = "dev"
VOLUME_NAME = "import"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

In [ ]:
# Create the import volume if it doesn't exist
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
print(f"Volume ready: {VOLUME_PATH}")
print(f"Upload your .parquet files to this volume before running the next cells.")

In [ ]:
# List available parquet files in the import volume
import os

parquet_files = [f for f in os.listdir(VOLUME_PATH) if f.endswith('.parquet')]
print(f"Found {len(parquet_files)} parquet files in {VOLUME_PATH}:")
for f in parquet_files:
    file_path = os.path.join(VOLUME_PATH, f)
    size = os.path.getsize(file_path)
    print(f"  - {f}: {size / (1024*1024):.2f} MB")

In [ ]:
# Import each parquet file as a Delta table
imported = []
failed = []

for parquet_file in parquet_files:
    table_name = parquet_file.replace('.parquet', '')
    full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    file_path = f"{VOLUME_PATH}/{parquet_file}"

    try:
        # Read parquet file
        df = spark.read.parquet(file_path)
        row_count = df.count()

        # Write as Delta table with Change Data Feed enabled
        df.write \
            .format("delta") \
            .option("delta.enableChangeDataFeed", "true") \
            .mode("overwrite") \
            .saveAsTable(full_table_name)

        imported.append((table_name, row_count))
        print(f"Imported {table_name} ({row_count} rows) -> {full_table_name}")
    except Exception as e:
        failed.append((table_name, str(e)))
        print(f"Failed to import {table_name}: {e}")

print(f"\nImported {len(imported)} tables, {len(failed)} failed")

In [ ]:
# Verify imported tables
print("Verifying imported tables:\n")
for table_name, expected_count in imported:
    full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    actual_count = spark.sql(f"SELECT COUNT(*) FROM {full_table_name}").collect()[0][0]
    status = "OK" if actual_count == expected_count else "MISMATCH"
    print(f"  {table_name}: {actual_count} rows [{status}]")

In [ ]:
# Summary
print("=" * 50)
print("IMPORT SUMMARY")
print("=" * 50)
print(f"\nSuccessfully imported {len(imported)} tables:")
for table_name, row_count in imported:
    print(f"  {CATALOG}.{SCHEMA}.{table_name}: {row_count} rows")

if failed:
    print(f"\nFailed to import {len(failed)} tables:")
    for table_name, error in failed:
        print(f"  {table_name}: {error}")

print(f"\nNext step: Run the vector_search_setup notebook to create search indices.")